# 10 · Security & PII
Column masks, region row filter and PII tags on Gold (module 10). Run **once, after** the job has built Gold; masks survive later rebuilds because Gold uses `INSERT OVERWRITE`.

Free Edition has no account groups, so roles come from `security.user_roles` keyed on `current_user()`.
⚠️ Delete any test roles before running the job, otherwise the row filter applies to the job and the Gold guardrails fail.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
sec, g = f"{catalog}.security", f"{catalog}.gold"

def run(*stmts):
    for stmt in stmts:
        spark.sql(stmt)

In [ ]:
# Roles via mapping tables
run(
    f"CREATE SCHEMA IF NOT EXISTS {sec} COMMENT 'Access-control mapping tables and functions'",
    f"""CREATE TABLE IF NOT EXISTS {sec}.user_roles (
          UserEmail STRING NOT NULL,
          Role      STRING NOT NULL)
        COMMENT 'Who holds which HCM role (pii_reader | hcm_admin | hrbp). Stands in for account groups on Free Edition.'""",
    f"""CREATE TABLE IF NOT EXISTS {sec}.hrbp_region_access (
          UserEmail STRING NOT NULL,
          Region    STRING NOT NULL)
        COMMENT 'Which regions an HR business partner may see.'""",
    f"""CREATE OR REPLACE FUNCTION {sec}.has_role(p_role STRING)
        RETURNS BOOLEAN
        RETURN EXISTS (SELECT 1 FROM {sec}.user_roles
                       WHERE UserEmail = current_user() AND Role = p_role)""",
)

In [ ]:
# Column masks on dimworker
run(
    f"""CREATE OR REPLACE FUNCTION {g}.mask_full_name(v STRING)
        RETURNS STRING
        RETURN CASE WHEN {sec}.has_role('pii_reader') THEN v ELSE concat(left(v, 1), '*****') END""",
    f"""CREATE OR REPLACE FUNCTION {g}.mask_date_of_birth(v DATE)
        RETURNS DATE
        RETURN CASE WHEN {sec}.has_role('pii_reader') THEN v ELSE make_date(year(v), 1, 1) END""",
    f"""CREATE OR REPLACE FUNCTION {g}.mask_national_id(v STRING)
        RETURNS STRING
        RETURN 'XXX-XX-XXXX'""",
    f"ALTER TABLE {g}.dimworker ALTER COLUMN FullName           SET MASK {g}.mask_full_name",
    f"ALTER TABLE {g}.dimworker ALTER COLUMN DateOfBirth        SET MASK {g}.mask_date_of_birth",
    f"ALTER TABLE {g}.dimworker ALTER COLUMN NationalIdentifier SET MASK {g}.mask_national_id",
)

In [ ]:
# Row filter: HRBPs see only their region
run(
    f"""CREATE OR REPLACE FUNCTION {g}.rf_by_region(location_key BIGINT)
        RETURNS BOOLEAN
        RETURN {sec}.has_role('hcm_admin')
            OR {sec}.has_role('pii_reader')
            OR NOT {sec}.has_role('hrbp')
            OR EXISTS (
                 SELECT 1
                 FROM {g}.dimlocation l
                 JOIN {sec}.hrbp_region_access m ON m.Region = l.Region
                 WHERE l.LocationKey = location_key
                   AND m.UserEmail = current_user())""",
    f"ALTER TABLE {g}.factworkforceheadcount SET ROW FILTER {g}.rf_by_region ON (LocationKey)",
    f"ALTER TABLE {g}.factworkforceattrition SET ROW FILTER {g}.rf_by_region ON (LocationKey)",
)

In [ ]:
# Tags and classification
run(
    f"ALTER TABLE {g}.dimworker ALTER COLUMN FullName           SET TAGS ('pii' = 'tier1')",
    f"ALTER TABLE {g}.dimworker ALTER COLUMN DateOfBirth        SET TAGS ('pii' = 'tier1')",
    f"ALTER TABLE {g}.dimworker ALTER COLUMN NationalIdentifier SET TAGS ('pii' = 'tier1')",
    f"ALTER TABLE {catalog}.silver.hcmworker        SET TAGS ('domain' = 'hcm', 'contains_pii' = 'true')",
    f"ALTER TABLE {catalog}.silver.quarantine       SET TAGS ('domain' = 'hcm', 'contains_pii' = 'true')",
    f"ALTER TABLE {catalog}.bronze.per_all_people_f SET TAGS ('domain' = 'hcm', 'contains_pii' = 'true')",
)
display(spark.sql(f"""SELECT schema_name, table_name, column_name, tag_value
                      FROM {catalog}.information_schema.column_tags WHERE tag_name = 'pii'"""))

## Try it
Run these in a SQL cell or editor (replace `workspace` if you use another catalog):
```sql
-- masked (no role)
SELECT PersonNumber, FullName, DateOfBirth, NationalIdentifier FROM workspace.gold.dimworker LIMIT 5;
-- unmasked names/DOB
INSERT INTO workspace.security.user_roles VALUES (current_user(), 'pii_reader');
-- APAC HRBP: only Pune, Chennai, Singapore at 2026-06-30
INSERT INTO workspace.security.user_roles VALUES (current_user(), 'hrbp');
INSERT INTO workspace.security.hrbp_region_access VALUES (current_user(), 'APAC');
-- clean up BEFORE running the job
DELETE FROM workspace.security.user_roles WHERE UserEmail = current_user();
DELETE FROM workspace.security.hrbp_region_access WHERE UserEmail = current_user();
```
If `SET MASK` / `SET ROW FILTER` is rejected in your workspace, use the secure-view fallback in module 10 §6.